In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import time

In [12]:
def get_naver_finance_news(days=3):
    # 1. 수집 대상 날짜 설정 (오늘 포함 3일치)
    target_date = datetime.now() - timedelta(days=days-1)
    target_date_str = target_date.strftime('%Y.%m.%d')
    
    base_url = "https://finance.naver.com/news/mainnews.naver"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    news_data = []
    page = 1
    stop_scraping = False

    print(f"{target_date_str} 이후의 기사를 수집합니다...")

    while not stop_scraping:
        params = {'page': page}
        response = requests.get(base_url, params=params, headers=headers)
        # 네이버 금융은 EUC-KR 인코딩을 주로 사용하므로 한글 깨짐 방지 설정
        response.encoding = 'euckr'
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 뉴스 리스트 영역 찾기
        news_list = soup.select('div.main_news > ul.realtimeNewsList > li') or \
                    soup.select('dl.newsList > dt, dl.newsList > dd')
        
        # 뉴스 리스트가 비어있으면 종료
        if not news_list:
            break
            
        # 실제 뉴스 항목은 dl 태그 안에 묶여 있음
        articles = soup.select('dl')
        
        for article in articles:
            # 제목과 링크 추출
            subject_tag = article.select_one('.articleSubject a')
            if not subject_tag: continue
            
            title = subject_tag.get_text().strip()
            link = "https://finance.naver.com" + subject_tag['href']
            
            # 언론사와 날짜 추출
            summary_tag = article.select_one('.articleSummary')
            press = summary_tag.select_one('.press').get_text().strip()
            wdate = summary_tag.select_one('.wdate').get_text().strip() # 형식: 2026.02.25 10:00
            
            # 날짜 비교를 위해 '2026.02.25' 부분만 추출
            article_date = wdate.split(' ')[0]
            
            # 설정한 기간보다 오래된 기사면 수집 중단
            if article_date < target_date_str:
                stop_scraping = True
                break
                
            news_data.append({
                '날짜': wdate,
                '언론사': press,
                '제목': title,
                '링크': link
            })
        
        print(f"현재 {page}페이지 수집 중... (누적: {len(news_data)}건)")
        page += 1
        time.sleep(0.3) # 서버 부하 방지를 위한 짧은 대기

    # 2. DataFrame 생성 및 CSV 내보내기
    df = pd.DataFrame(news_data)
    
    if not df.empty:
        filename = f"naver_finance_news_{datetime.now().strftime('%Y%m%d')}.csv"
        # 한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        print(f"--- 수집 완료! ---")
        print(f"총 {len(df)}건의 기사가 '{filename}'으로 저장되었습니다.")
    else:
        print("수집된 데이터가 없습니다.")
        
    return df

# 실행
df_news = get_naver_finance_news(days=3)

# 결과 확인
print(df_news.head())

2026.02.23 이후의 기사를 수집합니다...
수집된 데이터가 없습니다.
Empty DataFrame
Columns: []
Index: []
